# 航班价格监控 · EDA 02 — 定价结构、波动、告警噪音与售罄

拆解“为什么贵/便宜”：
1. **起飞日的日历效应** —— 星期几、月份、国庆窗口
2. **出发时刻 → 北京机场（大兴 vs 首都）→ 机型** 的价格差异
3. **航司价格分层**：谁是价格地板、谁常年最贵
4. **价格波动**：涨跌幅度、频率，是否临近起飞才剧烈
5. **告警噪音**：变价后是否常被“打脸”（反弹/回落）
6. **疑似售罄（删失）**：便宜航班通常在起飞前多久消失

全部对数据库只读。价格口径 = 抓取时该航班最低可订价快照。

## 0. 准备

In [ ]:
import sys, os, warnings
warnings.filterwarnings("ignore")
sys.path.insert(0, os.getcwd())
sys.path.insert(0, os.path.join(os.getcwd(), "eda"))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
import common as C

mpl.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "Noto Sans CJK SC", "DejaVu Sans"]
mpl.rcParams["axes.unicode_minus"] = False
mpl.rcParams.update({"figure.dpi": 110, "font.size": 10,
                     "axes.spines.top": False, "axes.spines.right": False,
                     "grid.color": "#e1e0d9", "grid.linewidth": 0.8,
                     "axes.grid": True, "axes.axisbelow": True})
PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100",
           "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
sns.set_palette(PALETTE)

df = C.load_flight_prices()
df = C.add_features(df)
grid = C.lead_grid(df)
floor = C.floor_by_lead(grid)
ts = C.trajectory_summary(df)
ev = C.change_events(df)
alerts = C.load_price_alerts()
logs = C.load_monitor_log()

DIRS = ["北京→泉州", "泉州→北京"]
DIR_COLOR = {"北京→泉州": PALETTE[0], "泉州→北京": PALETTE[1]}
WEEK = ["周一", "周二", "周三", "周四", "周五", "周六", "周日"]

print("加载完成:", len(df), "快照 |", len(grid), "grid |", len(ev), "变价事件 |",
      len(ts), "轨迹 |", len(alerts), "告警")

## 1. 起飞日的日历效应

用每个起飞日的“历史最低可订价”（该日所有航班所有时刻的 floor 最小值）当该日的价格水平。
同一把尺子下，星期几 / 哪个月 / 是否国庆窗口，贵多少？

### 1.1 每月 + 星期几
先造每个起飞日的“当日历史最低”（跨 lead 取 min），再按星期/月份聚合。

In [ ]:
dailymin = (floor.groupby(["route_label", "flight_date"])["floor"].min().rename("day_min")
            .reset_index())
dailymin["dow"] = pd.to_datetime(dailymin["flight_date"]).dt.dayofweek
dailymin["weekday"] = dailymin["dow"].map(dict(enumerate(WEEK)))
dailymin["month"] = pd.to_datetime(dailymin["flight_date"]).dt.month
dailymin["is_weekend"] = dailymin["dow"].isin([5, 6])

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
# 星期几
sub = dailymin[dailymin["route_label"].isin(DIRS)]
sns.boxplot(data=sub, x="weekday", y="day_min", hue="route_label", order=WEEK,
            palette=DIR_COLOR, ax=axes[0], width=.6, dodge=True)
axes[0].set_title("按起飞日星期几 · 当日最低可订价")
axes[0].set_xlabel("起飞日星期"); axes[0].set_ylabel("价格(元)")
# 月份
sns.boxplot(data=sub, x="month", y="day_min", hue="route_label",
            palette=DIR_COLOR, ax=axes[1], width=.6, dodge=True)
axes[1].set_title("按起飞日月份 · 当日最低可订价")
axes[1].set_xlabel("起飞月份"); axes[1].set_ylabel("价格(元)")
fig.tight_layout(); plt.show()

# 表格：周末 vs 工作日、中位价格
wk = (sub.groupby(["route_label", "is_weekend"])["day_min"]
      .agg(n="size", med="median", mean="mean").round(0))
wk.rename(index={False: "工作日", True: "周末"}, level=1, inplace=True)
wk

### 1.2 沿时间的价格曲线 + 国庆窗口
起飞日从 6/30 一路排到 10/1。标出周末与 9/25 之后（临近国庆）的区域，看价格是否爬坡。
每点 = 某起飞日“当日全网历史最低”。

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)
for ax, rl in zip(axes, DIRS):
    d = dailymin[dailymin["route_label"] == rl].sort_values("flight_date")
    d["dt"] = pd.to_datetime(d["flight_date"])
    wd = d["dt"].dt.dayofweek.isin([5, 6])
    ax.scatter(d["dt"][~wd], d["day_min"][~wd], s=14, color=DIR_COLOR[rl],
               alpha=.6, label="工作日")
    ax.scatter(d["dt"][wd], d["day_min"][wd], s=16, color="#eda100",
               alpha=.8, marker="D", label="周末")
    # 国庆前一周窗口 9/25–10/1
    holi = (d["dt"] >= "2026-09-25") & (d["dt"] <= "2026-10-01")
    ax.axvspan(pd.Timestamp("2026-09-25"), pd.Timestamp("2026-10-01"),
               color="#e34948", alpha=.10)
    ax.annotate("国庆窗口", xy=(pd.Timestamp("2026-10-01"),
                                d["day_min"].min() + 30), fontsize=9, color="#e34948")
    ax.set_title(f"{rl}")
    ax.set_ylabel("当日最低可订价(元)")
    ax.legend(fontsize=8)
    ax.xaxis.set_major_formatter(mpl.dates.DateFormatter("%m-%d"))
axes[0].set_xlabel("起飞日")
fig.tight_layout(); plt.show()

# 国庆窗口 vs 8月平日 的中位对比
for rl in DIRS:
    d = dailymin[dailymin["route_label"] == rl]
    d["dt"] = pd.to_datetime(d["flight_date"])
    holi = d[(d["dt"] >= "2026-09-25") & (d["dt"] <= "2026-10-01")]["day_min"]
    aug = d[(d["dt"] >= "2026-08-01") & (d["dt"] <= "2026-08-31")]["day_min"]
    print(f"{rl}: 国庆窗口(n={len(holi)}) 中位 {holi.median():.0f} | "
          f"8月平日(n={len(aug)}) 中位 {aug.median():.0f}")

1.结论：日历效应——**周中/周末差异很小**（周末中位仅 +30 元左右：去程 400→430、回程 350→385），箱体几乎重叠；**月份的差别才是主效应**：7/8/9 月当日最低稳定在 350–430 元平台，**国庆窗口（9/25–10/1）中位 590/610 元**（vs 8 月平日 420/390 元），**9/30–10/2 单日冲到 1,100–1,400 元**（图中点 10 月箱体 1200–1300+，9/30 为周三、10/1 周四，与周几无关）.→ 平日出行买点从容；节日出行提价是整体性、提前约 4 天才开始出现的“全窗口抬升”。

## 2. 出发时刻 → 机场 → 机型的定价
注意这些因素相互交织（早班 vs 晚班、大兴 vs 首都、不同航司/机型），
以下只是**分层看**，为后续做多因素模型提供线索。

### 2.1 出发时刻段

In [ ]:
slot_order = ["早(<9)", "上午(9-12)", "中午(12-14)", "下午(14-18)", "晚(≥18)"]
sub = df[df["route_label"].isin(DIRS) & (df["lead_days"] >= 0)]
fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=True)
for ax, rl in zip(axes, DIRS):
    s = sub[sub["route_label"] == rl]
    sns.boxplot(data=s, x="dep_slot", y="price", order=slot_order,
                color="#2a78d6", width=.55, ax=ax, showfliers=False)
    med = s.groupby("dep_slot")["price"].median().reindex(slot_order)
    ax.plot(range(len(slot_order)), med, "-o", color="#0b0b0b", ms=4)
    ax.set_title(rl)
    ax.set_xlabel("出发时刻段"); ax.set_ylabel("价格(元)")
    ax.tick_params(axis="x", rotation=45)
axes[0].set_xlabel("出发时刻段")
fig.tight_layout(); plt.show()

### 2.2 北京侧机场：大兴 vs 首都
北京⇄泉州线上同时有大兴（多中联航/联营低舱）与首都 T3/T2 的航班。
同一起飞日里，两边价差多大？

In [ ]:
sub = df[(df["route_label"].isin(DIRS)) & (df["lead_days"] >= 0)
         & df["bj_airport"].isin(["大兴", "首都"])]
ap_med = (sub.groupby(["route_label", "bj_airport"])["price"]
          .agg(med="median", mean="mean", n="size").round(0))
ap_med

fig, ax = plt.subplots(figsize=(8, 4))
sns.boxplot(data=sub, x="bj_airport", y="price", hue="route_label",
            palette=DIR_COLOR, ax=ax, showfliers=False)
ax.set_title("按北京侧机场 · 价格分布")
ax.set_xlabel("北京侧机场"); ax.set_ylabel("价格(元)")
fig.tight_layout(); plt.show()

# 同一起飞日的直接对照（限定两机场在同日都有快照）
sub["fd"] = pd.to_datetime(sub["flight_date"])
pt = sub.pivot_table(index="fd", columns=["route_label", "bj_airport"],
                     values="price", aggfunc="min")
for rl in DIRS:
    if (rl, "大兴") in pt.columns and (rl, "首都") in pt.columns:
        d = pt[(rl, "大兴")] - pt[(rl, "首都")]
        d = d.dropna()
        if len(d):
            print(f"{rl}: 同日 大兴−首都(全网最低) 中位 {d.median():.0f} 元, "
                  f"大兴更便宜的日期占比 {100*(d<0).mean():.0f}% (n={len(d)})")

### 2.3 机型家族
C919 只在国航北京→厦门；787/737MAX 数量少。看机型大体落在哪个价位带（含航司混淆）。

In [ ]:
sub = df[df["route_label"].isin(DIRS) & (df["lead_days"] >= 0)].copy()
sub["机型"] = sub["aircraft_type"].map(C.ac_family)
fam = (sub.groupby(["route_label", "机型"])["price"]
       .agg(med="median", n="size").round(0).reset_index())
fam["机型占比%"] = (100 * fam["n"] / fam.groupby("route_label")["n"].transform("sum")).round(1)
fam[fam["机型占比%"] > 1].sort_values(["route_label", "机型"])

2.结论：① **时刻**：去程“上午 9–12 点”最贵（中位 ~730 元，比早班贵 ~200 元）；回程从早到晚递减（早班 680 → 晚班 530）——非单调，但“晚班较便宜、上午热门班贵”在两方向都成立。② **机场**：同一起飞日“大兴−首都”的全网最低价差中位 **−50 元（去程）/−120 元（回程）**，大兴更便宜的日期占 **84%/100%**——从北京出发先看大兴场次能稳定省钱。③ **机型**：波音 737（63% 班次）中位 550–570 元 < 空客 321 580–590 < 空客 320 650 元，但机型与航司/时刻强混杂（见 §3），结论只作分层参考。

## 3. 航司价格分层：谁常年是地板、谁最贵

以“同一次爬取批次 × 同一起飞日”为单位：当日全网最低属于哪家航司（可并列）。
统计各航司“当最低价”的占比，以及相对当日最低的平均加价。

In [ ]:
unit = (df[(df["route_label"].isin(DIRS))]
        .groupby(["route_label", "flight_date", "crawl_time"])
        .agg(cheapest=("price", "min"))
        .reset_index())
joined = df[df["route_label"].isin(DIRS)].merge(
    unit, on=["route_label", "flight_date", "crawl_time"])
joined["is_cheapest"] = (joined["price"] == joined["cheapest"])
joined["gap"] = joined["price"] - joined["cheapest"]

share = (joined.groupby(["route_label", "airline"])["is_cheapest"]
         .mean().mul(100).round(1).rename("当最低价占比%").reset_index())
avg_gap = (joined[joined["gap"] > 0]
           .groupby(["route_label", "airline"])["gap"].median().round(0)
           .rename("相对当日最低的中位加价").reset_index())
ladder = share.merge(avg_gap, on=["route_label", "airline"], how="left")
ladder = ladder.sort_values(["route_label", "当最低价占比%"], ascending=[True, False])
ladder

3.结论：价格分层清晰——**北京→泉州：河北航空 68.9% 的批次当“全网最低价”（中位加价仅 50 元），是这条线的“价格守门员”**；中联航 23.8%（加价 100）、深航 11.6%（加价 200）、南航/厦航 ≤2.3%（加价 160–190）。**泉州→北京：中联航 26.1% + 河北 18.8% 领先**；厦航（加价 210）与深航（加价 230）几乎不提供最低价。→ 想买最低价优先盯 河北/中联航 的低舱；高价航司除非时刻刚需。

> 解读提示：占比最高 = 这条线上的“价格守门员”（常放低舱）；中位加价高且占比低 = 高价航司，
> 除非时刻刚需否则很少是最优。北京→厦门航司更多、C919/国航高占比，单独在 §5 售罄看它的特殊性。

## 4. 价格波动：涨跌的幅度、频率、时机
口径提醒：临期（lead≤3）的上涨中混有“只剩全价舱”的幅度跳变（见 00 §4b），
会抬高 1-3 天/起飞当天桶的平均幅度——该桶读作“上限参考”而非典型涨幅。


In [ ]:
# 4.1 涨跌幅度分布（去掉 0）
moved = ev[ev["diff"] != 0]
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
ax = axes[0]
sns.histplot(moved["diff"].abs(), bins=45, color=PALETTE[0], ax=ax)
ax.set_title("变价绝对幅度分布")
ax.set_xlabel("|价格变化|(元)"); ax.set_ylabel("事件数")
ax = axes[1]
sns.histplot(moved["diff"], bins=80, color=PALETTE[0], ax=ax)
ax.axvline(0, color="#0b0b0b", lw=.8)
ax.set_title("变价方向与幅度（>0涨 / <0跌）")
ax.set_xlabel("价格变化(元)"); ax.set_ylabel("事件数")
fig.tight_layout(); plt.show()

# 幅度 vs lead 时机（越小越临近起飞）
m2 = moved.copy()
m2["lead_bin"] = m2["lead_at"].map(C.lead_bin)
m2 = m2[m2["lead_bin"].isin(C.LEAD_BIN_ORDER)]
m2["adiff"] = m2["diff"].abs()
mag = (m2.groupby(["lead_bin", "direction"])["adiff"]
       .median().unstack().reindex(
           [b for b in C.LEAD_BIN_ORDER if b in m2["lead_bin"].values]))
print("—— 各 lead 桶 |Δ|中位(元)，按涨/跌 ——")
print(mag.round(0).to_string())
down_med = moved.loc[moved["diff"] < 0, "diff"].abs().median()
up_med = moved.loc[moved["diff"] > 0, "diff"].median()
print("\n总变价中位数 |Δ|=%.0f 元；下跌中位 %.0f / 上涨中位 %.0f"
      % (moved["diff"].abs().median(), down_med, up_med))

# 频率：每条轨迹变几次
n_changes = ev.groupby("traj_key").size().rename("n_changes")
print("\n—— 每条轨迹变价次数分布 ——")
print(n_changes.describe(percentiles=[.25, .5, .75, .9]).round(1).to_string())

4.结论：变价幅度中位 **90 元**（峰在 50–150 元、90% 在 280 元内），涨跌对称；**越临近起飞涨幅越大**：14–20 天上涨中位 90 → 1–3 天 160 → 起飞当天 **300 元**（下跌端稳定在 90–110 元）——临期主要是“向上跳价”。频率：每条轨迹平均 **7.3 次变价**（中位 7、90% ≤13 次），价格“跳档式”变化而非连续微调。注意 1–3 天/起飞当天桶含全价跳变（00 §4b），读作上限参考。

## 5. 告警噪音：变价被“打脸”的概率

思路：若一次下跌在 7 天内又涨回下跌前价格以上（**V 型反弹**），或一次上涨在 7 天内回落到上涨前以下
（**虚涨**），说明这次变价不稳定、按它下单容易被反向打脸——这就是告警的“噪音面”。
用真实 event 序列在轨迹内部判定。

In [ ]:
# 在每条轨迹内部（change_events 已按时间序输出），找“2 次后续变价内反向回到起点价以上/以下”的事件
rec_rows = []
for key, g in ev.groupby("traj_key", sort=False):
    arr = g[["from_price", "to_price", "direction"]].to_numpy()
    for i in range(len(arr) - 1):
        frm, to, dr = arr[i]
        if dr == "down":
            # 后续 2 次变价内是否出现 to_price >= frm（V 型反弹）
            for j in range(i + 1, min(i + 3, len(arr))):
                if arr[j][1] >= frm:
                    rec_rows.append({"key": key, "dir": "down", "recover": True})
                    break
            else:
                rec_rows.append({"key": key, "dir": "down", "recover": False})
        else:
            for j in range(i + 1, min(i + 3, len(arr))):
                if arr[j][1] <= frm:
                    rec_rows.append({"key": key, "dir": "up", "recover": True})
                    break
            else:
                rec_rows.append({"key": key, "dir": "up", "recover": False})
rec = pd.DataFrame(rec_rows)
if len(rec):
    summ = (rec.groupby("dir")["recover"]
            .agg(n="size", 反弹率=lambda x: 100 * x.mean()).round(1))
    print("—— 各方向变价的‘反弹率’（随后 2 次变价内反向，越高=越像噪音震荡）——")
    print(summ.to_string())
else:
    print("（事件过少，未计算）")

5.结论：**告警噪音面不小**——下跌后“随后 2 次变价内反弹回原价以上”的概率 **58.0%**（n=5,108），上涨后回落“打回原价以下”的概率 **42.3%**（n=4,700）。也就是说：过半的“降价”是震荡而非趋势、四成“涨价”会回落。→ 触发式告警（“跌了就买/涨了就追”）需要用**“低于同期历史分位 + 目标执行价”双条件过滤**，或对“反弹率高的方向”降低单次变价的置信度。

> “反弹率”读作“这类变价在接下来 2 次内大概率被反向”。若某方向反弹率很高，
> 说明大量变价是**震荡**而非趋势 —— 触发“下跌就买/上涨就追”的告警里，有一部分会反转。
> 需要提醒：这里用“至多 2 次事件”近似 7 天窗口，且没区分中间价；结论是**量级参考**而非精确率。

## 5b. price_alerts 告警表本身长什么样
注意口径：87 条告警**全部为 泉州→北京**，时间窗 8/6–8/21（由 `config.ALARM` 的配置窗口决定），
与“全航向的变价事件”不是同一统计范围——做“告警 vs 事件”对照时只代表该方向该窗口。


In [ ]:
if len(alerts):
    print("告警条数:", len(alerts), "| 变动额(元)均值 %.0f / 中位 %.0f"
          % (alerts["change_amount"].abs().mean(), alerts["change_amount"].abs().median()))
    print("变动% 分布:")
    print(alerts["change_percent"].describe(percentiles=[.25, .5, .75, .9]).round(1).to_string())
    print("\n降/涨价次数:", int((alerts["change_amount"] < 0).sum()),
          "/", int((alerts["change_amount"] > 0).sum()))
    # 告警是否都集中在临期？
    if "flight_date" in alerts.columns:
        alerts["dt"] = pd.to_datetime(alerts["alert_time"]).dt.normalize()
        alerts["fd"] = pd.to_datetime(alerts["flight_date"])
        alerts["lead"] = (alerts["fd"] - alerts["dt"]).dt.days
        print("\n告警发生时距起飞天数:", alerts["lead"].describe(
            percentiles=[.25, .5, .75]).round(0).to_string())
else:
    print("price_alerts 为空")

5b.结论：87 条告警：变动额均值 64 元/中位 70 元，51 次降价 + 36 次涨价，变动% 中位 −2.3%（极端 +54.8%/-27.9%）；**告警发生时距起飞 19–38 天**（中位 26 天）。重要口径：87 条**全部为 泉州→北京、时间窗 8/6–8/21**（config.ALARM 配置窗口所致），不代表全航向的告警行为——它与 §5 的“全航向反弹率”不可直接对比。

## 6. 疑似售罄（删失）推断

在 `trajectory_summary` 里我们把“同一航向同一起飞日还有别家在后来的爬取里出现、而它提前消失、
且当时尚未起飞”的轨迹标成 `censored` —— 极可能是**该航班已售罄/停止售卖**。
便宜是否先卖光？通常在起飞前多久消失？

**口径 caveat**：删失判定依赖“同航向下其它航班仍在被爬”这一前提——
- 北京→厦门是 `config.py` 里的 `alert_only` 航线，爬取在 **8/21 后整体停止**，其“提前消失”很可能是
  监控停止而非售罄 → 本节统计**只保留 北京⇄泉州 主线**（厦门另列，仅供参照）；
- 观察期内有 6 个整天空档（见 00 §3，含 7/27–7/28 连续两天），会让个别轨迹满足“别人还在爬”，
  产生少量伪删失 → 结论读作**量级参考**。

In [ ]:
# 主线（北京⇄泉州）为主统计；北京→厦门 单列（alert_only，8/21 停止爬取）
c = ts[ts["censored"]].copy()
c_main = c[c["route_label"].isin(DIRS)]
n_main = ts[ts["route_label"].isin(DIRS)]
print("疑似售罄轨迹数(主线):", len(c_main), "/", len(n_main),
      f"({100*len(c_main)/len(n_main):.1f}%)")
print("北京→厦门 单独:", len(c[c["route_label"] == "北京→厦门"]),
      "条 —— alert_only 航线，8/21 后停止爬取，不计入主线统计")
print("\n主线：消失时距起飞天数分布:")
print(c_main["soldout_lead"].describe(percentiles=[.25, .5, .75]).round(0).to_string())

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
# 消失前距离
axes[0].hist(c_main["soldout_lead"], bins=range(0, 35, 2), color="#1baf7a",
             edgecolor="white")
axes[0].set_title("疑似售罄：消失时距起飞还有几天（主线）")
axes[0].set_xlabel("距起飞天数"); axes[0].set_ylabel("轨迹数")
# 谁消失：按航司（主线）
air_c = c_main["airline"].value_counts().head(8)
axes[1].bar(air_c.index, air_c.values, color="#1baf7a")
axes[1].tick_params(axis="x", rotation=45)
axes[1].set_title("疑似售罄轨迹按航司（主线）")
axes[1].set_ylabel("轨迹数")
fig.tight_layout(); plt.show()

# 售罄前价格 vs 未售罄全程最低——便宜先走？（主线）
comp = pd.concat([c_main[["price_min"]].assign(type="疑似售罄"),
                  n_main[~n_main["censored"]][["price_min"]].assign(type="未售罄/保留")])
sns.boxplot(data=comp, x="type", y="price_min", palette=["#e34948", "#1baf7a"],
            width=.4)
plt.title("全程最低价分布：疑似售罄 vs 未售罄（便宜班次是否更易售罄？）")
plt.ylabel("该轨迹全程最低价(元)")
plt.tight_layout(); plt.show()

# 方向分布（含厦门参考）
print("\n按航向看疑似售罄数（含厦门参考）:")
print(c["route_label"].value_counts().to_string())

6.结论：主线（北京⇄泉州 1,572 条轨迹）疑似售罄 **56 条（3.6%）**：消失时距起飞中位 **20 天**（25–75 分位 16–25 天、最长 29 天），另有 6 条在 3–6 天急消失。按航司：**厦门航空 36 条、河北航空 17 条**最多——低价班次正是最先消失的（疑似售罄轨迹全程最低价中位约 450 元 vs 未售罄 520 元，且未售罄长尾到 2,100 元）。→ **便宜仓位确实先卖光**：想卡最低价要提前约 3 周、不要等临期；北京→厦门的 10 条因 alert_only 停爬未计入（见本节 caveat）。

## 小结（notebook 02）
- **日历**：周中/周末、月末假期窗口对当日最低价有可见影响；国庆(9/25–10/1) 是明显高峰。
- **机场**：同一起飞日大兴侧通常明显低于首都侧 → 从北京走，先看大兴方向的航班能省钱。
- **航司**：少数组“地板”航司常年占最低价（占比高、加价小），高价航司除非时刻刚需不必考虑。
- **波动/告警**：涨跌中位幅度约 90–100 元/次且大致均衡；若“反弹率”读数高，则对
  “跌了就买”要保留余量，避免追在一次假跌破上。
- **售罄**：主线（北京⇄泉州）删失轨迹的全程最低价分布整体偏左 → **便宜仓位确实更快卖光**；
  想卡最低价，不宜等到临期——临期剩下的多是未删失的贵舱或次优时刻。

三个 notebook（00 总览/质量、01 购买时机、02 结构·波动·售罄）合起来，
可支撑“某个目标出行日该提前多久、盯哪几个航班/机场、设多少心理价位”的决策框架。
